# Wine Reviews Analysis

Wine Enthusiast reviews dataset from [Kaggle (zynicide/wine-reviews)](https://www.kaggle.com/datasets/zynicide/wine-reviews).

- **Points**: Reviewer scores (80–100)
- **Price**: USD
- **Variety, country, winery**: Key categorical dimensions

## Setup & Data Loading

In [1]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from IPython.display import display, HTML

# Prism palette for all Plotly figures (discrete + continuous stops from same hues)
PRISM = list(px.colors.qualitative.Prism)
px.defaults.color_discrete_sequence = PRISM
px.defaults.color_continuous_scale = [
    [i / (len(PRISM) - 1), c] for i, c in enumerate(PRISM)
]
PRISM_DIVERGING = [[0, PRISM[10]], [0.5, "#f0f0f0"], [1, PRISM[2]]]
RATING_COLORS = {"unpopular": PRISM[0], "mid_tier": PRISM[4], "popular": PRISM[8]}

# Kaggle: Plotly iframe renderer + fig.show() — https://www.kaggle.com/code/stpeteishii/solution-to-a-plotly-graph-cannot-be-displayed
_IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()
if _IS_KAGGLE:
    pio.renderers.default = "iframe"


def show_plotly(fig):
    if os.environ.get("PLOTLY_FORCE_HTML", "").lower() in ("1", "true", "yes"):
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    elif _IS_KAGGLE:
        fig.show()
    else:
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


# Resolve data path: Kaggle first, then local (same pattern as denver-cpi)
KAGGLE_PATHS = [
    Path("/kaggle/input/datasets/zynicide/wine-reviews/winemag-data-130k-v2.csv"),
]
LOCAL_PATH = Path("data/winemag-data-130k-v2.csv")

csv_path = next((p for p in KAGGLE_PATHS if p.exists()), None)
if csv_path is not None:
    print(f"Using Kaggle path: {csv_path}")
else:
    csv_path = LOCAL_PATH if LOCAL_PATH.exists() else Path.cwd() / "data" / "winemag-data-130k-v2.csv"
    print(f"Using local path: {csv_path}")

df_raw = pd.read_csv(csv_path)


Using local path: data/winemag-data-130k-v2.csv


In [2]:
df_raw.info()
df_raw.head()

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             129971 non-null  int64  
 1   country                129908 non-null  str    
 2   description            129971 non-null  str    
 3   designation            92506 non-null   str    
 4   points                 129971 non-null  int64  
 5   price                  120975 non-null  float64
 6   province               129908 non-null  str    
 7   region_1               108724 non-null  str    
 8   region_2               50511 non-null   str    
 9   taster_name            103727 non-null  str    
 10  taster_twitter_handle  98758 non-null   str    
 11  title                  129971 non-null  str    
 12  variety                129970 non-null  str    
 13  winery                 129971 non-null  str    
dtypes: float64(1), int64(2), str(11)
memory usage: 

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


## Exploratory Analysis

In [3]:
# Points distribution
fig = px.histogram(df_raw, x="points", nbins=21, title="Distribution of Wine Points")
show_plotly(fig)

In [4]:
# Price vs Points (exclude outliers for readability)
df_plot = df_raw.dropna(subset=["price", "points"])
df_plot = df_plot[df_plot["price"] < 200]
fig = px.scatter(df_plot, x="price", y="points", color="points", title="Wine Price vs Points")
show_plotly(fig)

In [5]:
# Top countries by review count
country_counts = df_raw["country"].value_counts().head(15).reset_index()
country_counts.columns = ["country", "count"]
fig = px.bar(country_counts, x="country", y="count", title="Reviews by Country")
show_plotly(fig)

## Distribution of Points by Factor

In [6]:
# Points distribution by country (top 12 by review count)
top_countries = df_raw["country"].value_counts().head(12).index.tolist()
df_country = df_raw[df_raw["country"].isin(top_countries)]
fig = px.box(df_country, x="country", y="points", title="Distribution of Points by Country", color="country")
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
show_plotly(fig)

In [7]:
# Points distribution by winery (top 15 by review count)
top_wineries = df_raw["winery"].value_counts().head(15).index.tolist()
df_winery = df_raw[df_raw["winery"].isin(top_wineries)]
fig = px.box(df_winery, x="winery", y="points", title="Distribution of Points by Winery", color="winery")
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
show_plotly(fig)

## Best Ranked Wineries (Radar)

In [8]:
# Best ranked wineries: min 50 reviews, top 8 by avg points
winery_stats = (
    df_raw.dropna(subset=["price"])
    .groupby("winery")
    .agg(
        avg_points=("points", "mean"),
        avg_price=("price", "mean"),
        review_count=("points", "count"),
        pct_90plus=("points", lambda x: (x >= 90).mean() * 100),
    )
    .query("review_count >= 50")
    .sort_values("avg_points", ascending=False)
    .head(8)
)
# Normalize for radar (0-100 scale)
winery_stats["points_norm"] = (winery_stats["avg_points"] - 80) * 5  # 80->0, 100->100
winery_stats["price_norm"] = (winery_stats["avg_price"] / winery_stats["avg_price"].max() * 100).clip(0, 100)
winery_stats["count_norm"] = (winery_stats["review_count"] / winery_stats["review_count"].max() * 100).clip(0, 100)

fig = go.Figure()
for winery in winery_stats.index:
    row = winery_stats.loc[winery]
    fig.add_trace(
        go.Scatterpolar(
            r=[row["points_norm"], row["count_norm"], row["price_norm"], row["pct_90plus"], row["points_norm"]],
            theta=["Avg Points", "Review Volume", "Avg Price", "% 90+ pts", "Avg Points"],
            name=winery,
            fill="toself",
        )
    )
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title="Best Ranked Wineries (min 50 reviews, top 8 by avg points)",
    height=500,
)
show_plotly(fig)

## Points vs Price by Factor (with trendlines)

In [9]:
# Points vs price by country (top 8 countries, price < 150, OLS per country)
df_plot = df_raw.dropna(subset=["price", "points", "country"])
df_plot = df_plot[df_plot["price"] < 150]
top_c = df_plot["country"].value_counts().head(8).index.tolist()
df_plot = df_plot[df_plot["country"].isin(top_c)]
fig = px.scatter(
    df_plot, x="price", y="points", color="country",
    trendline="ols", trendline_scope="trace",
    title="Points vs Price by Country (line of best fit per country)",
)
show_plotly(fig)

In [10]:
# Points vs price by winery (top 8 wineries by review count, price < 150)
df_plot = df_raw.dropna(subset=["price", "points", "winery"])
df_plot = df_plot[df_plot["price"] < 150]
top_w = df_plot["winery"].value_counts().head(8).index.tolist()
df_plot = df_plot[df_plot["winery"].isin(top_w)]
fig = px.scatter(
    df_plot, x="price", y="points", color="winery",
    trendline="ols", trendline_scope="trace",
    title="Points vs Price by Winery (line of best fit per winery)",
)
show_plotly(fig)

## Popular vs Unpopular Wines

Define **popular** (high-rated) as points ≥ 90 and **unpopular** (low-rated) as points ≤ 84. We'll analyze which factors distinguish them.

In [11]:
# Define popular (≥90 pts) vs unpopular (≤84 pts); exclude middle range for cleaner contrast
df = df_raw.dropna(subset=["points", "price"]).copy()
df["rating_class"] = pd.cut(
    df["points"],
    bins=[0, 84, 90, 101],
    labels=["unpopular", "middle", "popular"]
)
df_pop = df[df["rating_class"].isin(["popular", "unpopular"])].copy()
df_pop["is_popular"] = (df_pop["rating_class"] == "popular").astype(int)

print(f"Popular (≥90): {len(df_pop[df_pop['rating_class']=='popular']):,}")
print(f"Unpopular (≤84): {len(df_pop[df_pop['rating_class']=='unpopular']):,}")
df_pop[["points", "price", "country", "variety", "rating_class"]].head(10)

Popular (≥90): 31,030
Unpopular (≤84): 11,832


,points,price,country,variety,rating_class
119,92,80.0,France,Riesling,popular
120,92,70.0,Italy,Nebbiolo,popular
121,92,36.0,US,Chardonnay,popular
122,92,39.0,US,Zinfandel,popular
123,92,40.0,Australia,Shiraz-Cabernet Sauvignon,popular
124,92,45.0,US,Cabernet Sauvignon,popular
125,91,45.0,South Africa,Cabernet Sauvignon,popular
126,91,48.0,France,Gewürztraminer,popular
127,91,13.0,France,White Blend,popular
128,91,17.0,France,Pinot Blanc,popular


### Single-variate analysis: what differs between popular and unpopular wines?

In [12]:
# 1. PRICE: Popular wines cost more
df_pv = df_pop[df_pop["price"] < 200]  # cap for readable viz
fig = px.histogram(
    df_pv, x="price", color="rating_class",
    nbins=40, barmode="overlay", opacity=0.6,
    title="Price distribution: Popular vs Unpopular"
)
fig.update_layout(xaxis_title="Price (USD)")
show_plotly(fig)

stats = df_pop.groupby("rating_class")["price"].agg(["mean", "median", "count"])
display(stats.round(2))

,mean,median,count
rating_class,,,
unpopular,18.76,15.0,11832
popular,60.34,48.0,31030


In [13]:
# 2. COUNTRY: Share of popular vs unpopular by country (top 12)
top_c = df_pop["country"].value_counts().head(12).index
df_c = df_pop[df_pop["country"].isin(top_c)]
ctab = pd.crosstab(df_c["country"], df_c["rating_class"], normalize="index")
ctab = ctab.sort_values("popular", ascending=False).reset_index()
ctab_m = pd.melt(ctab, id_vars=["country"], var_name="rating", value_name="share")
fig = px.bar(ctab_m, x="country", y="share", color="rating", barmode="stack", title="Country: share of popular vs unpopular")
fig.update_layout(xaxis_tickangle=-45, yaxis_tickformat=".0%")
show_plotly(fig)

In [14]:
# 3. VARIETY: Avg points and popular share by variety (min 500 reviews)
var_stats = (
    df_pop.groupby("variety")
    .agg({"points": "mean", "is_popular": ["sum", "count"]})
    .assign(
        popular_pct=lambda x: x[("is_popular", "sum")] / x[("is_popular", "count")],
        count=lambda x: x[("is_popular", "count")]
    )
)
var_stats.columns = ["avg_points", "n_popular", "n", "popular_pct", "count"]
var_top = var_stats[var_stats["count"] >= 500].sort_values("popular_pct", ascending=False).head(15)

fig = px.bar(var_top.reset_index(), x="variety", y="popular_pct", title="Variety: % popular (≥90 pts), min 500 reviews")
fig.update_layout(xaxis_tickangle=-45, yaxis_tickformat=".0%")
show_plotly(fig)

In [15]:
# 4. PROVINCE: Top provinces by popular share (min 200 reviews)
prov_stats = (
    df_pop.groupby("province")
    .agg({"is_popular": ["sum", "count"]})
)
prov_stats.columns = ["n_popular", "n"]
prov_stats["popular_pct"] = prov_stats["n_popular"] / prov_stats["n"]
prov_top = prov_stats[prov_stats["n"] >= 200].sort_values("popular_pct", ascending=False).head(12)

fig = px.bar(prov_top.reset_index(), x="province", y="popular_pct", title="Province: % popular (≥90 pts), min 200 reviews")
fig.update_layout(xaxis_tickangle=-45, yaxis_tickformat=".0%")
show_plotly(fig)

### Multivariate analysis: joint effects and predictive factors

In [16]:
# 1. CORRELATION: points, price, is_popular (numeric)
corr_df = df_pop[["points", "price", "is_popular"]].corr()
fig = px.imshow(corr_df, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, title="Correlation matrix")
show_plotly(fig)

In [17]:
# 2. LOGISTIC REGRESSION: popular ~ price + country + variety
# One-hot encode top categories to avoid huge feature space
import statsmodels.api as sm
import numpy as np

df_model = df_pop.dropna(subset=["country", "variety"]).copy()
top_countries = df_model["country"].value_counts().head(10).index
top_varieties = df_model["variety"].value_counts().head(15).index
df_model["country_top"] = df_model["country"].where(df_model["country"].isin(top_countries), "Other")
df_model["variety_top"] = df_model["variety"].where(df_model["variety"].isin(top_varieties), "Other")

X = pd.get_dummies(df_model[["price", "country_top", "variety_top"]], drop_first=True)
X = X.astype(np.float64)
# Scale price for interpretability
X["price"] = (X["price"] - X["price"].mean()) / X["price"].std()
X = sm.add_constant(X)
y = df_model["is_popular"].values.astype(np.float64)

logit = sm.Logit(y, X).fit(disp=0)
print(logit.summary().tables[1])

                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const                              2.1026      0.134     15.659      0.000       1.839       2.366
price                              8.5782      0.102     84.193      0.000       8.378       8.778
country_top_Australia              1.6850      0.150     11.271      0.000       1.392       1.978
country_top_Austria                4.0731      0.211     19.305      0.000       3.660       4.487
country_top_Chile                  0.1971      0.138      1.425      0.154      -0.074       0.468
country_top_France                 1.7865      0.112     15.986      0.000       1.567       2.006
country_top_Germany                2.6562      0.263     10.105      0.000       2.141       3.171
country_top_Italy                  1.4295      0.122     11.723      0.000       1.191       1.669
country_to

In [18]:
# 3. CHI-SQUARED: Country and Variety vs popular (categorical independence)
from scipy.stats import chi2_contingency

top_c = df_pop["country"].value_counts().head(12).index
top_varieties = df_pop["variety"].value_counts().head(15).index

# Country
ctab_country = pd.crosstab(df_pop["country"].where(df_pop["country"].isin(top_c), "Other"), df_pop["rating_class"])
chi2, p, dof, expected = chi2_contingency(ctab_country)
print(f"Country vs rating: chi2={chi2:.1f}, p={p:.2e} (reject independence: p<0.05)")

# Variety (top varieties)
df_var = df_pop.assign(variety_top=df_pop["variety"].where(df_pop["variety"].isin(top_varieties), "Other"))
ctab_var = pd.crosstab(df_var["variety_top"], df_var["rating_class"])
chi2_v, p_v, _, _ = chi2_contingency(ctab_var)
print(f"Variety vs rating: chi2={chi2_v:.1f}, p={p_v:.2e}")

Country vs rating: chi2=4626.1, p=0.00e+00 (reject independence: p<0.05)
Variety vs rating: chi2=3320.5, p=0.00e+00


In [19]:
# 4. SCATTER: Price vs Points, colored by country (top 6)
df_s = df_pop[df_pop["price"] < 150]
top6 = df_s["country"].value_counts().head(6).index
df_s = df_s[df_s["country"].isin(top6)]

fig = px.scatter(
    df_s.sample(min(5000, len(df_s)), random_state=42),
    x="price", y="points", color="country", opacity=0.5,
    trendline="ols", trendline_scope="overall",
    title="Price vs Points by country (top 6); popular ≥90, unpopular ≤84"
)
show_plotly(fig)